In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 14 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251206_134342.csv
Loaded: NBA_DFS_20251206_134221.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Michael Porter Jr,Over,26.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
1,PrizePicks,player_points,Michael Porter Jr,Under,26.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
2,PrizePicks,player_points,Trey Murphy III,Over,20.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
3,PrizePicks,player_points,Trey Murphy III,Under,20.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21
4,PrizePicks,player_points,Jeremiah Fears,Over,16.5,-137,2025-12-06,2025-12-06T21:41:23Z,2025-12-06 13:42:21


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

# underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 64 players...
[MIN] No data found for Egor Demin
[MIN] No data found for Ron Holland
[MIN] No data found for A.J. Green
Found 61 valid players
Generated 1585 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,ODDS 1,ODDS 2,IMPLIED_PROB 1,IMPLIED_PROB 2,TOTAL_EDGE
839,Marvin Bagley III,Jaime Jaquez Jr.,13.5,14.5,under,over,7.05,22.29,6.45,7.79,100,-108,0.49,0.51,14.24
1282,Kevin Porter Jr.,Jaden McDaniels,20.5,13.5,under,over,14.42,19.39,6.08,5.89,-108,-106,0.51,0.50,11.97
278,Noah Clowney,Julius Randle,16.5,20.5,under,over,10.74,26.11,5.76,5.61,-130,-102,0.55,0.49,11.37
1577,Maxime Raynaud,Kevin Durant,10.5,26.5,under,under,5.19,21.32,5.31,5.18,-110,-115,0.51,0.52,10.49
203,Danny Wolf,Ryan Nembhard,10.5,9.5,under,under,5.50,4.54,5.00,4.96,-108,-115,0.51,0.52,9.96
105,Saddiq Bey,Nickeil Alexander-Walker,17.5,20.5,under,over,13.26,25.30,4.24,4.80,-118,-118,0.53,0.53,9.04
761,Kyshawn George,P.J. Washington,15.5,13.5,under,over,10.77,17.49,4.73,3.99,-128,-106,0.55,0.50,8.72
873,Darius Garland,Bam Adebayo,16.5,20.5,under,over,12.62,24.40,3.88,3.90,-105,-110,0.50,0.51,7.78
330,Bryce McGowens,Ivica Zubac,8.5,14.5,under,over,4.84,17.94,3.66,3.44,105,-105,0.48,0.50,7.10
1110,Myles Turner,Naz Reid,12.5,13.5,under,over,9.13,16.63,3.37,3.13,-108,-103,0.51,0.50,6.50


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)


prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 88 players...
[MIN] No data found for Egor Demin
[MIN] No data found for A.J. Green
Found 86 valid players
Generated 3138 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,ODDS 1,ODDS 2,IMPLIED_PROB 1,IMPLIED_PROB 2,TOTAL_EDGE
1343,Marvin Bagley III,Jaime Jaquez Jr.,13.5,14.5,under,over,7.05,22.29,6.45,7.79,100,-108,0.49,0.51,14.24
2120,Kevin Porter Jr.,Aaron Holiday,20.5,10.5,under,under,14.42,4.21,6.08,6.29,-108,-107,0.51,0.50,12.37
2771,Jaden McDaniels,Maxime Raynaud,13.5,10.5,over,under,19.39,5.19,5.89,5.31,-106,-110,0.50,0.51,11.20
2696,Julius Randle,Kevin Durant,20.5,26.5,over,under,26.11,21.32,5.61,5.18,-102,-115,0.49,0.52,10.79
573,Danny Wolf,Ryan Nembhard,10.5,9.5,under,under,5.50,4.54,5.00,4.96,-108,-115,0.51,0.52,9.96
348,Noah Clowney,Anthony Davis,15.5,20.5,under,under,10.74,15.56,4.76,4.94,100,-110,0.49,0.51,9.70
93,Trey Murphy III,Jalen Duren,20.5,18.5,over,under,25.03,13.90,4.53,4.60,-114,-120,0.52,0.53,9.13
1105,Nickeil Alexander-Walker,Andrew Wiggins,21.0,16.5,over,over,25.30,20.63,4.30,4.13,-137,-110,0.56,0.51,8.43
1531,Cam Whitmore,P.J. Washington,10.5,13.5,under,over,6.60,17.49,3.90,3.99,-104,-106,0.50,0.50,7.89
1912,Darius Garland,Bam Adebayo,16.5,20.5,under,over,12.62,24.40,3.88,3.90,-105,-110,0.50,0.51,7.78


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 64 players...
[MIN] No data found for Egor Demin
[MIN] No data found for Ron Holland
[MIN] No data found for A.J. Green
Found 61 valid players
Generated 22723 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,SIDE 1,SIDE 2,SIDE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,EDGE 1,EDGE 2,EDGE 3,ODDS 1,ODDS 2,ODDS 3,IMPLIED_PROB 1,IMPLIED_PROB 2,IMPLIED_PROB 3,TOTAL_EDGE
16303,Marvin Bagley III,Kevin Porter Jr.,Jaime Jaquez Jr.,13.5,20.5,14.5,under,under,over,7.05,14.42,22.29,6.45,6.08,7.79,100,-108,-108,0.49,0.51,0.51,20.32
6293,Noah Clowney,Jaden McDaniels,Maxime Raynaud,16.5,13.5,10.5,under,over,under,10.74,19.39,5.19,5.76,5.89,5.31,-130,-106,-110,0.55,0.50,0.51,16.96
4050,Danny Wolf,Julius Randle,Kevin Durant,10.5,20.5,26.5,under,over,under,5.50,26.11,21.32,5.00,5.61,5.18,-108,-102,-115,0.51,0.49,0.52,15.79
2321,Saddiq Bey,Nickeil Alexander-Walker,Ryan Nembhard,17.5,20.5,9.5,under,over,under,13.26,25.30,4.54,4.24,4.80,4.96,-118,-118,-115,0.53,0.53,0.52,14.00
15048,Kyshawn George,Bam Adebayo,P.J. Washington,15.5,20.5,13.5,under,over,over,10.77,24.40,17.49,4.73,3.90,3.99,-128,-110,-106,0.55,0.51,0.50,12.62


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 88 players...
[MIN] No data found for Egor Demin
[MIN] No data found for A.J. Green
Found 86 valid players
Generated 62832 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,SIDE 1,SIDE 2,SIDE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,EDGE 1,EDGE 2,EDGE 3,ODDS 1,ODDS 2,ODDS 3,IMPLIED_PROB 1,IMPLIED_PROB 2,IMPLIED_PROB 3,TOTAL_EDGE
37928,Marvin Bagley III,Jaime Jaquez Jr.,Aaron Holiday,13.5,14.5,10.5,under,over,under,7.05,22.29,4.21,6.45,7.79,6.29,100,-108,-107,0.49,0.51,0.50,20.53
53703,Kevin Porter Jr.,Jaden McDaniels,Maxime Raynaud,20.5,13.5,10.5,under,over,under,14.42,19.39,5.19,6.08,5.89,5.31,-108,-106,-110,0.51,0.50,0.51,17.28
16598,Danny Wolf,Julius Randle,Kevin Durant,10.5,20.5,26.5,under,over,under,5.50,26.11,21.32,5.00,5.61,5.18,-108,-102,-115,0.51,0.49,0.52,15.79
9714,Noah Clowney,Jalen Duren,Ryan Nembhard,15.5,18.5,9.5,under,under,under,10.74,13.90,4.54,4.76,4.60,4.96,100,-120,-115,0.49,0.53,0.52,14.32
2236,Trey Murphy III,Nickeil Alexander-Walker,Anthony Davis,20.5,21.0,20.5,over,over,under,25.03,25.30,15.56,4.53,4.30,4.94,-114,-137,-110,0.52,0.56,0.51,13.77
